In [0]:
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("destination_timestamp", "")
dbutils.widgets.text("source_file_format", "")

source_system        = dbutils.widgets.get("source_system")
table_name           = dbutils.widgets.get("table_name")
destination_timestamp = dbutils.widgets.get("destination_timestamp")
source_file_format   = dbutils.widgets.get("source_file_format")

In [0]:
source_path = f"/Volumes/adb_caresync/destination/destination_volume/{source_system}/{table_name}/{destination_timestamp}/"

# Check if the source path exists before attempting to read
try:
    dbutils.fs.ls(source_path)
except Exception:
    print(f"No files are received. Path does not exist: {source_path}")
    dbutils.notebook.exit("No files received – source path not found.")

if source_file_format.lower() == "csv":
    destination_df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(source_path)
else:
    destination_df = spark.read.format("parquet").load(source_path)

In [0]:
display(destination_df)

In [0]:
from pyspark.sql import functions as F

updated_df = destination_df.withColumn("destination_timestamp", F.lit(destination_timestamp)) \
    .withColumn("insert_timestamp", F.current_timestamp())

In [0]:
display(updated_df)

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")

In [0]:
%sql
select * from adb_caresync.bronze.hospitals